# Resting membrane potential analysis

Resting membrane potential ist determined from I=0 current-clamp recordings using this custom Python script (pyabf; Harden & Bhatt). For each .abf file, data were pooled across all sweeps and the median membrane potential was calculated per channel. Median was used in preference to mean to reduce sensitivity to transient noise artifacts. The script automatically detects single- and dual-channel recordings and exports results to a spreadsheet.

In [ ]:
import pyabf
import numpy as np
import pandas as pd
import os

In [ ]:
# Path to folder containing I=0 .abf files
directory = r"directory_path"
 
file_names = [f for f in os.listdir(directory) if f.endswith('.abf')]
 
results = []
 
for file in file_names:
    file_path = os.path.join(directory, file)
    abf = pyabf.ABF(file_path)
 
    if abf.channelCount == 1:
        # Single-channel: pool data across all sweeps
        data_ch0 = []
        for sweep in range(abf.sweepCount):
            abf.setSweep(sweepNumber=sweep, channel=0)
            data_ch0.extend(abf.sweepY)
 
        # Median is used instead of mean for robustness against transient noise artifacts
        median_ch0 = np.median(data_ch0)
        print(f"{file} | Ch0: {median_ch0:.3f} mV")
        results.append({'File': file, 'Median Ch0 (mV)': median_ch0, 'Median Ch1 (mV)': None})
 
    else:
        # Dual-channel: pool and compute median separately per channel
        data_ch0, data_ch1 = [], []
        for sweep in range(abf.sweepCount):
            abf.setSweep(sweepNumber=sweep, channel=0)
            data_ch0.extend(abf.sweepY)
            abf.setSweep(sweepNumber=sweep, channel=1)
            data_ch1.extend(abf.sweepY)
 
        median_ch0 = np.median(data_ch0)
        median_ch1 = np.median(data_ch1)
        print(f"{file} | Ch0: {median_ch0:.3f} mV | Ch1: {median_ch1:.3f} mV")
        results.append({'File': file, 'Median Ch0 (mV)': median_ch0, 'Median Ch1 (mV)': median_ch1})
 
# Single-channel files have None in the Ch1 column
df = pd.DataFrame(results)
df.to_excel('resting_potential_I0.xlsx', index=False)